# Bag-of-Words GRPO

Sample script for zero-step-sampling GRPO on the bag-of-words dataset.

The policy is a Gaussian around the model's scalar prediction,
$m_\theta(z\mid x) = \mathcal N(f_\theta(x), \sigma^2)$. Rollouts are samples from this policy; rewards are $-(z-y)^2$ against the noisy target; advantages are standardized within each rollout group.

In [1]:
import polars as pl
import torch

from src import get_repo_base
from src.experiments.bag_of_words.grpo import BagOfWordsGRPOConfig

from src.experiments.bag_of_words.analysis import (
    BagOfWordsAnalysisConfig,
    corr_expr,
    rsq_expr,
)

repo_root = get_repo_base()
device = torch.device("cuda:1")

/home/nlyu/Code/maxrl-statistics/src/experiments/bag_of_words/grpo.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


## Configure

In [ ]:
config = BagOfWordsGRPOConfig.get_canonical(
    dataset="homoskedastic",
    dataset_base_folder=repo_root / "artifacts" / "bow-data",
    study_base_folder=repo_root / "artifacts" / "bow-grpo-example",
    corr=0.2,
    aux_words_ratio=0.5,
    train_epochs=2,
    num_rollouts_per_sample=4,
    gaussian_stdev=1.0,
)

display(config.visualize())
print(f"Study folder: {config.study_folder}")
print(f"Dataset corr target: {config.data.corr:.4f}")
print(f"Backbone lr: {config.optimizer.lr:.3e}")
print(f"Head lr:     {config.optimizer.head_lr:.3e}")
print(f"Rollouts/sample: {config.num_rollouts_per_sample}")
print(f"Policy stdev:    {config.gaussian_stdev}")

state = config.initialize(device=device)
display(
    state.dataset.token_lengths_plot(
        filter_threshold=config.tokenization.filter_samples_above_n_tokens,
    )
)

In [4]:
state.run_training()

grpo epoch 0:   0%|          | 0/781 [00:00<?, ?it/s]

validation epoch 0:   0%|          | 0/390 [00:00<?, ?it/s]

grpo epoch 1:   0%|          | 0/781 [00:00<?, ?it/s]

validation epoch 1:   0%|          | 0/390 [00:00<?, ?it/s]

## Results

Training saves:
1. A compact `metrics.parquet` to disk which contains per-epoch sufficient statistics to compute metrics.
2. Validation parquet containing per-row ground-truth and target.

In [5]:
metrics_path = config.study_folder / "metrics.parquet"
metrics = pl.read_parquet(metrics_path)
metrics

epoch,train_target_xx,train_target_xy,train_target_yy,train_target_n,train_ground_truth_xx,train_ground_truth_xy,train_ground_truth_yy,train_ground_truth_n,val_target_xx,val_target_xy,val_target_yy,val_target_n,val_ground_truth_xx,val_ground_truth_xy,val_ground_truth_yy,val_ground_truth_n
i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
0,8974.287109,1022.167419,49554.207031,49984.0,8974.287109,1007.728027,1973.314941,49984.0,6332.562012,974.9198,49494.257812,49920.0,6332.562012,899.945923,1999.744507,49920.0
1,4474.662109,1827.929443,49550.347656,49984.0,4474.662109,1600.0625,1973.267822,49984.0,2384.549805,1403.647705,49494.257812,49920.0,2384.549805,1407.270996,1999.744507,49920.0


In [ ]:
analysis = BagOfWordsAnalysisConfig.from_grouped({"example": [(0, config.study_folder)]})

analysis.plot_vs_epoch([
    rsq_expr(split="train", y="ground_truth"),
    rsq_expr(split="val", y="ground_truth"),
    corr_expr(split="train", y="ground_truth"),
    corr_expr(split="val", y="ground_truth"),
])


In [8]:
last_epoch = int(metrics["epoch"].max())
validation_path = config.study_folder / str(last_epoch) / "validation.parquet"
validation_df = pl.read_parquet(validation_path)
validation_df.head()

model_preds,ground_truth,target
f64,f64,f64
-0.044678,0.015564,0.761719
-0.316406,-0.263672,-0.055908
-0.137695,0.0,-0.238281
0.031982,0.046631,-0.679688
-0.417969,-0.279297,0.376953
